# Resolver Workbench (Act-1)
? block ??????????LLM????????summary???

In [ ]:
import os, sys, time, json
from pathlib import Path
ROOT = Path(r'd:/Projects/7_rush/3_functional_query/1_claude')
WS = ROOT / 'workspace'
if str(WS / 'script') not in sys.path:
    sys.path.insert(0, str(WS / 'script'))
for line in (WS / '.env').read_text(encoding='utf-8').splitlines():
    line=line.strip()
    if (not line) or line.startswith('#') or '=' not in line:
        continue
    k,v = line.split('=',1)
    os.environ.setdefault(k.strip(), v.strip().strip('\"').strip("'"))
print('env loaded')

In [ ]:
from PxFquery import PxFquery

pxf = PxFquery()
pxf.load_data_dir(str(WS / 'output/store/gsea_anndata'))
# ?? always_llm???????LLM????
pxf.enable_resolver(
    index_dir=str(WS / 'output/store/query_index'),
    provider='minimax',
    model=os.getenv('MINIMAX_MODEL', 'MiniMax-M2.7'),
    use_fast_path=False,
    log_level='INFO',
)
resolver = pxf._resolver
print('resolver ready')

In [ ]:
# ?????????? + ??LLM????
q = 'In A549, what pathways are affected by EGFR knockdown?'
t0=time.perf_counter()
res = resolver.resolve_and_query(q, top_n=10, summarize=False)
dt=time.perf_counter()-t0
print('time_sec', round(dt,3))
print('hit_level', res.resolver_meta.get('hit_level'))
print('pert_type', res.resolver_meta.get('pert_type'))
print(json.dumps(res.resolver_meta.get('llm_call_stats', {}), ensure_ascii=False, indent=2))

In [ ]:
# ?? vs ??????
tests = [
    'In A549, what pathways change after erlotinib treatment?',
    'In A549, what pathways are affected by an EGFR inhibitor?',
    'In PC3, estimate pathway response for l-theanine-like perturbation.',
    'In A549, what pathways are affected by EGFR knockdown?',
]
rows=[]
for q in tests:
    t0=time.perf_counter()
    r = resolver.resolve_and_query(q, top_n=10, summarize=False)
    dt=time.perf_counter()-t0
    m=r.resolver_meta
    rows.append({
        'q': q,
        'pert_type': m.get('pert_type'),
        'selected_source': m.get('selected_source'),
        'hit_level': m.get('hit_level'),
        'time_sec': round(dt,3),
        'llm_calls': m.get('llm_call_stats',{}).get('query',{}),
    })
rows

In [ ]:
# Stepwise??????????
from PxFquery.index.cellline_index import CellLineIndex
idx = CellLineIndex(
    WS / 'output/store/query_index/cellline_index.json',
    WS / 'output/store/query_index/cellline_neighbors.json',
)
bio = 'non-small cell lung carcinoma'
trace=[]
def chooser(desc, options, level):
    sel = resolver._smart_choose_cell_node(desc, options, level)
    trace.append({'level': level, 'selected': sel, 'options': options})
    return sel
cells = idx.traverse(bio_context=bio, llm_choose_fn=chooser)
print('cells', cells[:10], '... total', len(cells))
trace

In [ ]:
# summary???????
q='In A549, what pathways are affected by EGFR knockdown?'
t0=time.perf_counter()
r=resolver.resolve_and_query(q, top_n=10, summarize=True)
dt=time.perf_counter()-t0
print('time_sec', round(dt,3))
print('summary_len', len(r.summary or ''))
print((r.summary or '')[:500])
print(json.dumps(r.resolver_meta.get('llm_call_stats',{}), ensure_ascii=False, indent=2))

In [ ]:
# ???????? M2.7-highspeed
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv('MINIMAX_API_KEY'),
    base_url=os.getenv('MINIMAX_BASE_URL', 'https://api.minimaxi.com/v1')
)
model='M2.7-highspeed'
t0=time.perf_counter()
resp=client.chat.completions.create(
    model=model,
    messages=[{'role':'user','content':'Reply with JSON: {\"ok\":true}'}],
    temperature=0,
    max_tokens=40,
)
dt=time.perf_counter()-t0
print('time_sec', round(dt,3))
print(resp.choices[0].message.content)